# T-CRL: Temporal Causal Representation Learning for Behavioral Outcome Prediction
EC523 Deep Learning — Boston University

In [ ]:
# install deps
!pip install scikit-learn seaborn --quiet

# fresh clone every time to avoid stale cached code
!rm -rf /content/EC523
!git clone https://github.com/ezheng05/EC523.git /content/EC523 --quiet

# clear any compiled bytecode
!find /content/EC523 -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null; true

import sys, os
sys.path.insert(0, '/content/EC523')

# mount drive for dataset access
from google.colab import drive
drive.mount('/content/drive')

# adjust this path if your dataset is in a different Drive folder
DATASET_ROOT = '/content/drive/MyDrive/globem-dataset-multi-year-datasets-for-longitudinal-human-behavior-modeling-generalization-1.1'

In [11]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from config.config import ModelConfig
from src.data.dataset import make_splits
from src.models.encoder import TCRL_Encoder, Baseline_Standard_Encoder
from src.models.vae import TCRL_BetaVAE
from src.training.trainer import train_epoch, evaluate
from src.utils.metrics import compute_regression_metrics, compute_auc_metrics, print_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')

device: cuda


In [12]:
cfg = ModelConfig()

cohort_dirs = [os.path.join(DATASET_ROOT, c) for c in cfg.cohorts]
for d in cohort_dirs:
    if not os.path.isdir(d):
        print(f'WARNING: missing cohort dir: {d}')

train_ds, val_ds, test_ds = make_splits(
    cohort_dirs, seq_len=cfg.seq_len,
    val_ratio=cfg.val_ratio, test_ratio=cfg.test_ratio
)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.batch_size, num_workers=2, pin_memory=True)

loading cohorts...
  INS-W_1
  INS-W_2


KeyError: ['CESD_10items_POST', 'STAIS_POST', 'CESD_10items_PRE', 'STAIS_PRE']

In [ ]:
from src.data.dataset import BASELINE_COLS

feat_dim = train_ds.feat_dim
ehr_dim  = len(BASELINE_COLS)

enc  = TCRL_Encoder(feat_dim=feat_dim, seq_len=cfg.seq_len, ehr_dim=ehr_dim, hidden_dim=cfg.hidden_dim, dropout=cfg.dropout)
tcrl = TCRL_BetaVAE(enc, latent_dim=cfg.latent_dim, num_targets=cfg.num_targets).to(device)
opt  = optim.Adam(tcrl.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

train_losses, val_losses = [], []
best_val, patience_count = float('inf'), 0

print('--- training t-crl ---')
for epoch in range(cfg.epochs):
    stats = train_epoch(tcrl, train_loader, opt, device, cfg.beta, cfg.lambda_sparsity)
    train_losses.append(stats['task'])
    if (epoch + 1) % 10 == 0:
        vp, vt = evaluate(tcrl, val_loader, device)
        val_mse = float(((vp - vt) ** 2).mean())
        val_losses.append(val_mse)
        print(f'epoch {epoch+1:03d}/{cfg.epochs} | task: {stats["task"]:.4f} | val mse: {val_mse:.4f}', flush=True)
        if val_mse < best_val:
            best_val = val_mse
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= cfg.early_stop_patience:
                print(f'early stop at epoch {epoch+1}')
                break

In [ ]:
b_enc  = Baseline_Standard_Encoder(feat_dim=feat_dim, seq_len=cfg.seq_len, ehr_dim=ehr_dim, hidden_dim=cfg.hidden_dim, dropout=cfg.dropout)
bmodel = TCRL_BetaVAE(b_enc, latent_dim=cfg.latent_dim, num_targets=cfg.num_targets).to(device)
b_opt  = optim.Adam(bmodel.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

b_train_losses, b_val_losses = [], []
b_best_val, b_patience = float('inf'), 0

print('--- training baseline (no missingness gate) ---')
for epoch in range(cfg.epochs):
    stats = train_epoch(bmodel, train_loader, b_opt, device, cfg.beta, cfg.lambda_sparsity)
    b_train_losses.append(stats['task'])
    if (epoch + 1) % 10 == 0:
        vp, vt = evaluate(bmodel, val_loader, device)
        val_mse = float(((vp - vt) ** 2).mean())
        b_val_losses.append(val_mse)
        print(f'epoch {epoch+1:03d}/{cfg.epochs} | task: {stats["task"]:.4f} | val mse: {val_mse:.4f}', flush=True)
        if val_mse < b_best_val:
            b_best_val = val_mse
            b_patience = 0
        else:
            b_patience += 1
            if b_patience >= cfg.early_stop_patience:
                print(f'early stop at epoch {epoch+1}')
                break

In [ ]:
print('=== test results ===')
for name, model in [('t-crl', tcrl), ('baseline', bmodel)]:
    y_pred, y_true = evaluate(model, test_loader, device)
    reg = compute_regression_metrics(y_pred, y_true)
    auc = compute_auc_metrics(y_pred, y_true)
    print(f'\n{name}:')
    print_metrics(reg, auc)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(train_losses, label='t-crl train')
axes[0].plot(b_train_losses, label='baseline train')
axes[0].set_title('training task loss')
axes[0].set_xlabel('epoch')
axes[0].set_ylabel('mse')
axes[0].legend()

x_ticks = list(range(9, cfg.epochs, 10))
axes[1].plot(x_ticks, val_losses, label='t-crl val')
axes[1].plot(x_ticks, b_val_losses, label='baseline val')
axes[1].set_title('validation mse')
axes[1].set_xlabel('epoch')
axes[1].set_ylabel('mse')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, (name, model) in zip(axes, [('t-crl', tcrl), ('baseline', bmodel)]):
    adj = model.adj.detach().cpu().numpy()
    sns.heatmap(adj, annot=True, cmap='coolwarm', center=0, fmt='.3f', ax=ax)
    ax.set_title(f'{name} causal adjacency')
    ax.set_xlabel('effect (latent dim)')
    ax.set_ylabel('cause (latent dim)')

plt.tight_layout()
plt.show()

In [ ]:
from src.utils.metrics import TARGET_NAMES

y_pred, y_true = evaluate(tcrl, test_loader, device)
y_pred, y_true = y_pred.numpy(), y_true.numpy()

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
axes = axes.flatten()
for i, (ax, name) in enumerate(zip(axes, TARGET_NAMES)):
    ax.scatter(y_true[:, i], y_pred[:, i], alpha=0.5, s=20)
    lo = min(y_true[:, i].min(), y_pred[:, i].min())
    hi = max(y_true[:, i].max(), y_pred[:, i].max())
    ax.plot([lo, hi], [lo, hi], 'r--', linewidth=1)
    ax.set_xlabel('actual')
    ax.set_ylabel('predicted')
    ax.set_title(name)

plt.suptitle('t-crl: predicted vs actual (test set)', y=1.01)
plt.tight_layout()
plt.show()